# Fine-tune ReactionT5 on a larger ORD sample (Model 1, Kaggle GPU)

Same `scripts/train_reactant_model_ord.py` as the Colab notebook (`colab/05_train_reactant_ord.ipynb`), pointed at a bigger training pool (~150,000 ORD reactions instead of 60,000) to test whether more data pushes accuracy past the v2 result (ORD exact_match 50.7%/top-5 74.3%).

**Two bugs fixed after the first 150k attempt (see `RESULTS.md` sections 5-6):**
1. That run launched without `--no-augment`, silently inheriting the script's default SMILES augmentation (prob=0.5), which independent testing (variant 4) already showed hurts accuracy despite a healthy-looking `eval_loss` curve.
2. The *corrected* (no-augment) rerun still only matched v2's 60k result (48.7% vs 50.7% top-1) rather than beating it -- because it used the script's *current* defaults (`--learning-rate` 2e-5, `--num-train-epochs` 2), not v2's actual proven config. Confirmed from v2's own saved `training_args.bin`: v2 used `lr=5e-5` and `3` epochs. Those defaults were lowered later, during v3/v4's augmentation-overfitting debugging, and never restored for the plain no-augmentation case -- so "more data" was never cleanly tested against the training regimen that actually worked.

The cell below now passes `--no-augment --learning-rate 5e-5 --num-train-epochs 3` explicitly.

**Also fixed: actually using both GPUs.** Plain `python` with 2 GPUs visible makes HF `Trainer` fall back to `torch.nn.DataParallel` (overhead, no reliable speedup -- confirmed on the 250k Kaggle run: lower throughput than a single T4 on Colab, plus the tell-tale `"Was asked to gather along dimension 0..."` warning). The cell below launches via `torchrun --nproc_per_node=2` for proper `DistributedDataParallel`: each GPU runs its own process, and the effective batch size doubles (16/GPU x 2 GPUs).

**Run this as Save & Run All (Commit), not an interactive Draft Session.** The first attempt lost ~40% of its progress when an interactive Draft Session reset on tab reload/idle -- Kaggle doesn't reliably keep Draft Sessions alive without an active connection. Commit mode runs as a background batch job independent of the browser and reliably persists `/kaggle/working` as the version's Output.

**Before running:** in the notebook Settings panel (right sidebar) turn on **Internet** and **GPU accelerator** (T4x2). Kaggle's free GPU quota is **30 hours/week**.

**Data:** the 150k-reaction sample was built locally (`build_train_data_ord.py --pool-count 150000`, same seed/eval-exclusion logic as the 60k pool, so it's still leak-free against `data/v2_ord_eval_targets.json`) and must be uploaded as a **Kaggle Dataset** (two files: `reactants_train.jsonl`, `reactants_val.jsonl`), then added to this notebook as an input (`+ Add Input` in the right sidebar).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))
if torch.cuda.device_count() < 2:
    print("WARNING: fewer than 2 GPUs visible -- the torchrun --nproc_per_node=2 launch below expects 2.")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

**Input data.** Adjust the dataset slug below to match whatever you named the Kaggle Dataset you uploaded (visible under `/kaggle/input/` once added as an input).

In [ ]:
import os

train_file = "/kaggle/input/retro-planner-ord-150k/reactants_train.jsonl"  # @param {type:"string"}
val_file = "/kaggle/input/retro-planner-ord-150k/reactants_val.jsonl"  # @param {type:"string"}

assert os.path.exists(train_file), f"Not found: {train_file} -- did you add the dataset as an input (+ Add Input, right sidebar)?"
assert os.path.exists(val_file), f"Not found: {val_file}"
print("Train file:", train_file, "--", sum(1 for _ in open(train_file)), "rows")
print("Val file:", val_file, "--", sum(1 for _ in open(val_file)), "rows")

**Cross-session resume on Kaggle.** There's no Drive-style live mount here -- `/kaggle/working` only persists once you **Save Version** ("commit") the notebook, which turns its contents into this notebook's own Output, downloadable as a dataset. To continue training in a later session:

1. This session: train, then **Save Version** before your quota/time runs out. The committed `/kaggle/working/<output_dir_name>` becomes an Output you can download or directly reuse.
2. Next session: either (a) add *this same notebook's* previous Output version as an input (Kaggle lets you pick a specific version's output), or (b) download the `final`/`checkpoint-N` folder and re-upload it as its own small Dataset -- same idea as the Colab notebook's cross-account resume.
3. Point `resume_from_checkpoint_path` below at wherever that folder landed under `/kaggle/input/...`.

Leave `resume_from_checkpoint_path` blank for a first run.

In [ ]:
resume_from_checkpoint_path = ""  # @param {type:"string"}
# e.g. /kaggle/input/model1-ord150k-checkpoint/checkpoint-4750  (full Trainer checkpoint -- exact resume)
# or   /kaggle/input/model1-ord150k-checkpoint/final           (weights only -- fresh optimizer/step count)

In [ ]:
output_dir = "/kaggle/working/model1_reactant_ord150k_v2cfg"  # @param {type:"string"}
time_budget_minutes = 240  # @param {type:"number"}
# 3 epochs (matching v2's proven config) over 147k/32 effective-batch is ~13,782 steps; at the
# ~63.5 steps/min DDP rate observed on the previous 150k Kaggle T4x2 run, that's ~3.6h -- 240 min
# (4h) leaves headroom. output_dir renamed (_v2cfg) so it can't be confused with the earlier,
# weaker-hyperparameter 150k run's local download.

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"
resume_flag = ["--resume-from-checkpoint", resume_from_checkpoint_path] if resume_from_checkpoint_path else []

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate 5e-5 \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(resume_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Same log-redirect reasoning as the Colab notebook: printing per-step output directly in the cell can make the tab unresponsive over a multi-hour run. Since this runs as a Commit job, you don't need to watch it at all -- check back later via `kaggle kernels status <user>/<slug>` (CLI) or the Output tab.

`--local-work-dir` points at `/kaggle/temp` (fast local scratch disk, wiped between sessions) so Trainer's own checkpoint rotation never touches `/kaggle/working` directly; the training script's own `DriveSyncCallback`-style logic still copies out one `latest_checkpoint` folder under `output_dir` after every save. Only rank 0 (of the 2 `torchrun` processes) does this Drive-style sync and the final save (fixed via `RANK`-based rank detection -- see `scripts/train_reactant_model_ord.py`), so there's no risk of the two GPU processes racing to write the same files.

**When done:** download via CLI (`kaggle kernels output <user>/<slug> -p <dest>`) or the Output tab. `output_dir/final` has the model. Evaluate it exactly like the other checkpoints:

```
python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets.json \
    --t5-model <downloaded_final_dir> \
    --num-beams 10 --output experiments/v2_model1_topk/ord150k_v2cfg_topk.json
```